In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import copy
import pandas as pd

# ========================== DEVICE ==========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ========================== DATASET ==========================
class DeepfakeDataset(Dataset):
    def __init__(self, img_dir, labels=None, transform=None):
        self.img_dir = img_dir
        self.labels = labels
        self.transform = transform
        self.files = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_id = int(fname.split(".")[0])
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        if self.labels is not None:
            label = self.labels[img_id]
            return img, label
        return img, fname


# ========================== TRANSFORMS ==========================
mean = torch.tensor([0.5192, 0.4276, 0.3844])
std = torch.tensor([0.2860, 0.2640, 0.2635])

transform_rule = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05), # Added saturation and hue
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5), # Added RandomAffine
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

# ========================== LOAD DATA ==========================
train_dir = "dataset/train_images"
csv_dir = "dataset/train_solution.csv"

# Загрузка меток
#labels = {}
#with open(csv_dir, "r") as file:
#    for line in file:
#        line = line.strip()
#        if "," in line:
#            idx, val = line.split(",")
#            labels[int(idx)] = int(val)
#        else:
#            labels[int(line)] = 0

# Load labels using pandas for easier handling
df = pd.read_csv(csv_dir)
labels = df.set_index('Id')['target_feature'].to_dict()


# Разделение на train/val
all_ids = list(labels.keys())
all_labels = list(labels.values())  # Use list to prevent indexing errors

train_ids, val_ids, train_labels, val_labels = train_test_split(
    all_ids, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

# Datasets
train_dataset = DeepfakeDataset(train_dir, labels=labels, transform=transform_rule)
val_dataset = DeepfakeDataset(train_dir, labels=labels, transform=transform_rule)

train_dataset.files = [f"{i}.jpg" for i in train_ids]
val_dataset.files = [f"{i}.jpg" for i in val_ids]

# Calculate weights for WeightedRandomSampler
class_counts = np.bincount(train_labels)
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[i] for i in train_labels]

# WeightedRandomSampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True  # or False, depending on your needs
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    #shuffle=True, # No shuffle when using a sampler
    sampler=weighted_sampler, # Use the sampler
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

print(f"DataLoaders created successfully!")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# ========================== MODEL ==========================
class DeepBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ) if in_c != out_c else nn.Sequential()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class DeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = DeepBlock(3, 32)
        self.layer2 = DeepBlock(32, 64)
        self.layer3 = DeepBlock(64, 128)
        self.layer4 = DeepBlock(128, 256)
        self.layer5 = DeepBlock(256, 512)

        self.pool = nn.AvgPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.pool(self.layer1(x))
        x = self.pool(self.layer2(x))
        x = self.pool(self.layer3(x))
        x = self.pool(self.layer4(x))
        x = self.pool(self.layer5(x))
        x = self.global_pool(x)
        return self.classifier(x)

# ========================== TRAINING FUNCTION ==========================
def train_model(model, train_loader, val_loader, num_epochs=25):
    # Handling class weights directly in loss function
    class_counts = np.bincount(train_labels)
    weights = torch.tensor(class_counts / sum(class_counts), dtype=torch.float)
    class_weights = 1 / weights
    norm_weights = class_weights / class_weights.sum()    # Normalize to make sum = 1
    norm_weights = norm_weights.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight = norm_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )
    
    scaler = GradScaler()
    best_f1 = 0.0
    patience_counter = 0
    best_model_state = None
    MODEL_SAVE_DIR = "models"
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

    for epoch in range(num_epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_preds, train_true = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{num_epochs} [Train]")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            preds = outputs.argmax(dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_true.extend(labels.cpu().numpy())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Validation
        model.eval()
        val_loss = 0.0
        val_preds, val_true = [], []
        
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:2d}/{num_epochs} [Val  ]")
            for images, labels in pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())

        # Metrics
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_true, val_preds)
        val_f1 = f1_score(val_true, val_preds, average='binary')

        print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
            }, f"{MODEL_SAVE_DIR}/best_deepfake_model_f1_{val_f1:.4f}.pth")
            print(f"✅ Best model saved! F1 = {val_f1:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= 7:
            print("🛑 Early stopping!")
            break

        scheduler.step(avg_val_loss)

    model.load_state_dict(best_model_state)
    print(f"\nTraining finished! Best F1-score: {best_f1:.4f}")
    return model


# ========================== ЗАПУСК ==========================
model = DeepfakeCNN().to(DEVICE)
model = train_model(model, train_loader, val_loader, num_epochs=70)

# Сохранение финальной модели
torch.save(model.state_dict(), "models/final_deepfake_cnn.pth")
print("Final model saved!")


# ========================== INFERENCE ==========================
class InferenceDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.files = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_id = int(fname.split(".")[0])
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img, img_id


test_dir = "dataset/test_images"

inference_dataset = InferenceDataset(test_dir, transform=transform_rule)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

model.eval()
predictions = {}

with torch.no_grad():
    for images, ids in tqdm(inference_loader, desc="Inference"):
        images = images.to(DEVICE)
        outputs = model(images)

        # Apply sigmoid to get probabilities
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # Probability of class 1

        for i, img_id in enumerate(ids):
            predictions[img_id] = probs[i]


# Create the submission file
with open("submission.csv", "w") as f:
    f.write("Id,target_feature\n")
    for img_id in sorted(predictions.keys()):
        f.write(f"{img_id},{int(predictions[img_id] > 0.5)}\n") # Threshold at 0.5


print("Submission file created!")

Using device: cuda


C:\Users\Иван\AppData\Local\Temp\ipykernel_13336\2224670251.py:191: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


DataLoaders created successfully!
Train batches: 1250, Val batches: 313


Epoch  1/70 [Train]:   0%|          | 0/1250 [00:00<?, ?it/s]C:\Users\Иван\AppData\Local\Temp\ipykernel_13336\2224670251.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch  1/70 [Train]:   1%|          | 15/1250 [00:22<30:44,  1.49s/it, loss=0.4382]


KeyboardInterrupt: 